In [34]:
import ts05_wrapper as ts
import pandas as pd
import numpy as np
from figure_setup import figure_setup
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
figure_setup()

In [44]:
t = pd.Timestamp("2014-02-25 12:00:00", tz="UTC")
bx, by, bz = ts.ts05_field_at_time(t, 5.0, 1.0, 0.0)
print(bx, by, bz)

1.0455801079460185 0.7467456621861533 -8.027551482183775


In [82]:
t1 = pd.Timestamp("2014-02-25 00:00:00", tz="UTC")
t2 = t1 + pd.Timedelta(days=7)
tt = pd.date_range(start=t1, end=t2, freq=pd.Timedelta(hours=4))
tt

DatetimeIndex(['2014-02-25 00:00:00+00:00', '2014-02-25 04:00:00+00:00',
               '2014-02-25 08:00:00+00:00', '2014-02-25 12:00:00+00:00',
               '2014-02-25 16:00:00+00:00', '2014-02-25 20:00:00+00:00',
               '2014-02-26 00:00:00+00:00', '2014-02-26 04:00:00+00:00',
               '2014-02-26 08:00:00+00:00', '2014-02-26 12:00:00+00:00',
               '2014-02-26 16:00:00+00:00', '2014-02-26 20:00:00+00:00',
               '2014-02-27 00:00:00+00:00', '2014-02-27 04:00:00+00:00',
               '2014-02-27 08:00:00+00:00', '2014-02-27 12:00:00+00:00',
               '2014-02-27 16:00:00+00:00', '2014-02-27 20:00:00+00:00',
               '2014-02-28 00:00:00+00:00', '2014-02-28 04:00:00+00:00',
               '2014-02-28 08:00:00+00:00', '2014-02-28 12:00:00+00:00',
               '2014-02-28 16:00:00+00:00', '2014-02-28 20:00:00+00:00',
               '2014-03-01 00:00:00+00:00', '2014-03-01 04:00:00+00:00',
               '2014-03-01 08:00:00+00:00', '2014-0

In [87]:
from tqdm import tqdm
x_GSE = np.linspace(-30, 15, 100)
z_GSE = np.linspace(-15, 15, 100)
X, Z = np.meshgrid(x_GSE, z_GSE)
xyz = np.column_stack([X.ravel(), np.zeros_like(X).ravel(), Z.ravel()])

df = []
for t in tqdm(tt):
    for i in range(len(xyz)):
        bx, by, bz = ts.ts05_field_at_time(t, xyz[i][0], xyz[i][1], xyz[i][2])
        df.append(pd.Series({'t': t, 'bx': bx, 'by': by, 'bz': bz, 'x_GSE': xyz[i][0], 'z_GSE': xyz[i][2], 'y_GSE': xyz[i][1]}))
df_xz = pd.DataFrame(df)
df_xz['b'] = np.sqrt(df_xz['bx']**2 + df_xz['by']**2 + df_xz['bz']**2)
df_xz.set_index('t', inplace=True)

100%|██████████| 43/43 [00:34<00:00,  1.24it/s]


In [88]:
from tqdm import tqdm
x_GSE = np.linspace(-30, 15, 100)
y_GSE = np.linspace(-15, 15, 100)
X, Y = np.meshgrid(x_GSE, y_GSE)
xyz = np.column_stack([X.ravel(), Y.ravel(), np.zeros_like(X).ravel()])

df = []
for t in tqdm(tt):
    for i in range(len(xyz)):
        bx, by, bz = ts.ts05_field_at_time(t, xyz[i][0], xyz[i][1], xyz[i][2])
        df.append(pd.Series({'t': t, 'bx': bx, 'by': by, 'bz': bz, 'x_GSE': xyz[i][0], 'z_GSE': xyz[i][2], 'y_GSE': xyz[i][1]}))
df_xy = pd.DataFrame(df)
df_xy['b'] = np.sqrt(df_xy['bx']**2 + df_xy['by']**2 + df_xy['bz']**2)
df_xy.set_index('t', inplace=True)

100%|██████████| 43/43 [00:34<00:00,  1.24it/s]


In [ ]:
import imageio.v2 as imageio
import shutil
from pathlib import Path
tbins = df_xy.index.unique()
frames = []
path = Path('maps')
shutil.rmtree(path, ignore_errors=True)
path.mkdir(exist_ok=True)
for i, t in enumerate(tbins):
    fig, ax  = plt.subplots(2, 1, figsize=(14, 10))

    gr = df_xz.loc[t]
    B  = gr['b'].values.reshape(X.shape)
    Bx = gr['bx'].values.reshape(X.shape)
    Bz = gr['bz'].values.reshape(X.shape)
    c = ax[0].pcolormesh(X, Z, B, cmap='jet', shading='auto', norm=LogNorm(vmin=1.0, vmax=500))
    fig.colorbar(c, ax=ax[0])
    ax[0].set_title(t)
    ax[0].set_xlabel('X [R_E]')
    ax[0].set_ylabel('Z [R_E]')

    gr = df_xy.loc[t]
    B  = gr['b'].values.reshape(X.shape)
    Bx = gr['bx'].values.reshape(X.shape)
    By = gr['by'].values.reshape(X.shape)
    c = ax[1].pcolormesh(X, Y, B, cmap='jet', shading='auto', norm=LogNorm(vmin=1.0, vmax=500))
    fig.colorbar(c, ax=ax[1])
    ax[1].set_title(t)
    ax[1].set_xlabel('X [R_E]')
    ax[1].set_ylabel('Y [R_E]')

    plt.tight_layout()
    plt.savefig(path / f'map_{i:04d}.png')
    plt.close()
    frames.append(imageio.imread(path / f'map_{i:04d}.png'))
imageio.mimsave('movie.gif', frames, fps=5, loop=0)